# YouTube → SRT (ruso) con faster-whisper

Notebook autocontenido. No depende del resto del repo Sta-RU.

**Qué hace:** descarga audio de una lista de links de YouTube, los transcribe a SRT en ruso con `large-v2`, y te entrega todo en un ZIP. Suena un ruidito cuando termina.

**Antes de correr:** `Runtime → Change runtime type → T4 GPU` (o cualquier GPU). Sin GPU también corre, pero mucho más lento.


## 1) Setup

In [ ]:
!pip install -q faster-whisper yt-dlp
!apt-get -qq install -y ffmpeg > /dev/null

import torch, os, shutil, zipfile, time
from pathlib import Path

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "int8_float16" if DEVICE == "cuda" else "int8"
print(f"Device: {DEVICE}  |  compute_type: {COMPUTE_TYPE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


## 2) Pegá tus links de YouTube

Uno por línea. Admite videos sueltos y playlists. Las líneas que arrancan con `#` se ignoran.


In [ ]:
URLS_RAW = """
# https://www.youtube.com/watch?v=XXXXXXXXXXX
# https://www.youtube.com/watch?v=YYYYYYYYYYY
"""

URLS = [u.strip() for u in URLS_RAW.strip().splitlines() if u.strip() and not u.strip().startswith("#")]
assert URLS, "Pegá al menos un link y descomentá las líneas (sacale el #)."
print(f"{len(URLS)} link(s) a procesar:")
for u in URLS:
    print("  -", u)


## 3) Descargar audio

In [ ]:
import yt_dlp

AUDIO_DIR = Path("/content/audios")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

ydl_opts = {
    "outtmpl": str(AUDIO_DIR / "%(title).80s [%(id)s].%(ext)s"),
    "format": "bestaudio[ext=m4a]/bestaudio/best",
    "noplaylist": False,
    "ignoreerrors": True,
    "quiet": False,
    "no_warnings": True,
    "restrictfilenames": True,
}

with yt_dlp.YoutubeDL(ydl_opts) as ydl:
    ydl.download(URLS)

audios = sorted([p for p in AUDIO_DIR.iterdir() if p.is_file() and p.suffix.lower() in {".m4a", ".webm", ".opus", ".mp3", ".mp4", ".wav", ".ogg"}])
print(f"\nDescargados: {len(audios)}")
for a in audios:
    print("  -", a.name)


## 4) Transcribir a SRT (ruso, large-v2)

Parámetros equivalentes a tu llamada de PowerShell:

| PowerShell (faster-whisper-xxl) | Python (faster-whisper) |
|---|---|
| `--model large-v2` | `WhisperModel("large-v2")` |
| `--language ru` | `language="ru"` |
| `--compute_type int8_float16` | `compute_type="int8_float16"` |
| `--temperature 0` | `temperature=0` |
| `--beam_size 5` | `beam_size=5` |
| `--best_of 1` | `best_of=1` |
| `--task transcribe` | `task="transcribe"` |
| `--max_line_count 1 --sentence` | una línea por segmento al escribir SRT |
| `--max_line_width 200` | no se trunca (segmentos enteros) |

Si el SRT ya existe, salta (igual que tu script de PowerShell).


In [ ]:
from faster_whisper import WhisperModel

def fmt_ts(t):
    h = int(t // 3600)
    m = int((t % 3600) // 60)
    s = int(t % 60)
    ms = int(round((t - int(t)) * 1000))
    if ms == 1000:
        ms = 0
        s += 1
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

print("Cargando modelo large-v2...")
model = WhisperModel("large-v2", device=DEVICE, compute_type=COMPUTE_TYPE)
print("Modelo listo.\n")

total = len(audios)
t_global = time.time()
for i, audio in enumerate(audios, 1):
    srt = audio.with_suffix(".srt")
    if srt.exists():
        print(f"[{i}/{total}] SALTADO (ya existe): {audio.name}")
        continue

    print(f"[{i}/{total}] Procesando: {audio.name}")
    t0 = time.time()
    segments, info = model.transcribe(
        str(audio),
        language="ru",
        task="transcribe",
        temperature=0,
        beam_size=5,
        best_of=1,
        vad_filter=True,
        condition_on_previous_text=True,
    )
    print(f"   duración audio: {info.duration:.1f}s")

    n = 0
    with open(srt, "w", encoding="utf-8") as f:
        for seg in segments:
            text = seg.text.strip()
            if not text:
                continue
            n += 1
            f.write(f"{n}\n{fmt_ts(seg.start)} --> {fmt_ts(seg.end)}\n{text}\n\n")
    dt = time.time() - t0
    print(f"[{i}/{total}] Listo: {srt.name}  ({n} líneas, {dt:.1f}s)\n")

print(f"Finalizado. {total} videos procesados en {(time.time()-t_global)/60:.1f} min.")


## 5) Empaquetar SRTs en ZIP + ruidito + descarga

In [ ]:
from IPython.display import Audio, display
import numpy as np
from google.colab import files

srts = sorted(AUDIO_DIR.glob("*.srt"))
print(f"SRTs a empaquetar: {len(srts)}")
for s in srts:
    print("  -", s.name)

ZIP_PATH = "/content/subtitulos_ru.zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for s in srts:
        zf.write(s, arcname=s.name)
print(f"\nZIP listo: {ZIP_PATH}  ({os.path.getsize(ZIP_PATH)/1024:.1f} KB)")

# Ruidito: fanfarria ascendente sol-do-mi-sol-do
sr = 22050
out = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    env = np.exp(-3*t)
    out = np.concatenate([out, (0.3*env*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out, rate=sr, autoplay=True))

# Disparar la descarga al navegador
files.download(ZIP_PATH)
